# Chapter 4 — Instruction-Style Response Formatting

**Goal:** Teach the model to respond in a specific persona and format — not just *what* to say, but *how* to say it.

Chapters 2 and 3 taught the model output patterns (label, JSON). This chapter teaches **behavioral style**: tone, structure, and conversational conventions. This is what people mean by "instruction tuning."

**Task:** Answer customer support questions in this style:
- Start with a friendly, empathetic acknowledgment
- Answer with 2-3 bullet points
- End with *"Is there anything else I can help with?"*

## 4.1 Load the Base Model

In [ ]:
import torch, random
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)

print(f"Model loaded on {device}")

## 4.2 Create Training Data

Each example is `Query: <customer question>\nResponse:` → a multi-line response in the support style. We generate pairs programmatically to cover varied question types.

In [ ]:
random.seed(42)

def make_example(query, response):
    prompt = f"Query: {query}\nResponse:"
    completion = f"\n{response}"
    return {"prompt": prompt, "completion": completion}

topics = [
    ("how do I reset my password",
     ["I understand how frustrating it is to be locked out of your account.",
      "No worries at all — resetting your password is quick and easy."],
     ["Go to Settings > Account > Change Password",
      "Click the 'Forgot Password' link on the login page",
      "Check your email for the reset link (valid for 1 hour)"]),
    ("can I change my subscription plan",
     ["Absolutely — you're in control of your plan at any time.",
      "Great question! Switching plans is straightforward."],
     ["Visit Billing > Subscription in your account settings",
      "You can upgrade, downgrade, or switch between monthly and yearly",
      "Changes take effect at the start of your next billing cycle"]),
    ("my payment was charged twice this month",
     ["I'm sorry to hear about the duplicate charge — let's get this sorted out.",
      "That definitely shouldn't happen. Let me help you resolve this."],
     ["Check Billing > History to confirm both charges are from us",
      "If both are ours, contact billing@company.com with the transaction IDs",
      "Refunds are typically processed within 3-5 business days"]),
    ("how do I delete my account",
     ["I'm sorry to hear you're thinking of leaving — is there anything we can improve?",
      "We'd hate to see you go, but I'll make sure the process is smooth for you."],
     ["Go to Settings > Account > Delete Account at the bottom of the page",
      "You'll need to confirm via email for security purposes",
      "All your data will be permanently deleted after 30 days"]),
    ("the app is running very slow",
     ["I understand how annoying slowdowns can be — let's get this fixed.",
      "Let's troubleshoot the performance issues you're experiencing."],
     ["Clear the app cache: Settings > Storage > Clear Cache",
      "Make sure you're on the latest version — check your app store for updates",
      "Try closing background apps to free up memory"]),
    ("can I use the app offline",
     ["Great question! Here's how offline mode works.",
      "Yes, you can absolutely use the app without an internet connection."],
     ["Enable offline mode in Settings > Offline Access",
      "Recent documents are automatically cached for offline use",
      "Changes sync automatically when you reconnect to the internet"]),
    ("how do I share a file with my team",
     ["Collaboration is what we do best — let me show you how.",
      "Sharing with your team is simple and secure."],
     ["Click the Share button (top right) on any file or folder",
      "Enter team member emails or copy the shareable link",
      "Set permissions: View Only, Comment, or Edit access"]),
    ("is my data secure on your platform",
     ["Security is our top priority — I'm glad you asked.",
      "Absolutely. Here's a breakdown of how we protect your data."],
     ["All data is encrypted at rest using AES-256 and in transit via TLS 1.3",
      "We are SOC 2 Type II certified and GDPR compliant",
      "Two-factor authentication is available and strongly recommended"]),
]

raw_data = []

for question, greetings, bullets in topics:
    for _ in range(18):  # 8 topics × 18 = 144 total
        greeting = random.choice(greetings)
        chosen = random.sample(bullets, min(2, len(bullets)))
        bullet_text = "\n".join(f"- {b}" for b in chosen)
        closing = "Is there anything else I can help with?"
        response = f"{greeting}\n\n{bullet_text}\n\n{closing}"
        raw_data.append(make_example(question, response))

random.shuffle(raw_data)
split = int(0.85 * len(raw_data))
train_data = raw_data[:split]
eval_data = raw_data[split:]

print(f"Train: {len(train_data)}  |  Eval: {len(eval_data)}")
print(f"\nSample prompt:\n{train_data[0]['prompt']}")
print(f"\nSample completion:\n{train_data[0]['completion'][:200]}...")

## 4.3 Tokenize

Same masking approach as before. Completion length is longer here so we bump `max_length`.

In [ ]:
def tokenize(example):
    prompt_tokens = tokenizer(example["prompt"], truncation=True, max_length=256, padding=False)
    completion_tokens = tokenizer(example["completion"], truncation=True, max_length=192, padding=False)

    input_ids = prompt_tokens["input_ids"] + completion_tokens["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_tokens["input_ids"]) + completion_tokens["input_ids"] + [tokenizer.eos_token_id]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataset = Dataset.from_list(train_data).map(tokenize, remove_columns=["prompt", "completion"])
eval_dataset = Dataset.from_list(eval_data).map(tokenize, remove_columns=["prompt", "completion"])

print(f"Tokenized {len(train_dataset)} train / {len(eval_dataset)} eval examples")
s = train_dataset[0]
masked = sum(1 for l in s["labels"] if l == -100)
print(f"Sample: {len(s['input_ids'])} tokens, {masked} masked, {len(s['input_ids']) - masked} trained")

## 4.4 Apply LoRA

In [ ]:
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 4.5 Train

Longer outputs need more epochs. The model must learn not just the answer, but the entire response structure.

In [ ]:
def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(x["attention_mask"] + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

training_args = TrainingArguments(
    output_dir="./chapters/04-instruction/checkpoint",
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    learning_rate=5e-4,
    weight_decay=0.01,
    fp16=(device == "cuda"),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

print("Starting training...")
trainer.train()

## 4.6 Evaluate

We test on **unseen questions** — these were not in the training data. This tests whether the model learned the *style* or just memorized the 10 specific questions.

In [ ]:
def respond(query):
    prompt = f"Query: {query}\nResponse:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full.split("Response:")[-1].strip() if "Response:" in full else full

# Unseen questions — none of these were in training
test_queries = [
    "how do I update my email address",
    "is there a family sharing plan available",
    "my notifications are not working properly",
    "how can I export all my data at once",
]

for query in test_queries:
    print(f"{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    print(respond(query))
    print()

## 4.7 Evaluate Style Compliance

Check if the model consistently follows the three style rules on the test queries.

In [ ]:
test_queries = [
    "how do I update my email address",
    "is there a family sharing plan available",
    "my notifications are not working properly",
    "how can I export all my data at once",
    "can I merge two accounts together",
    "what browsers do you support",
    "how do I set up two-factor authentication",
    "is there a dark mode option",
]

scores = {"greeting": 0, "bullets": 0, "closing": 0}
total = len(test_queries)

for query in test_queries:
    response = respond(query)
    lines = response.strip().split("\n")

    # Rule 1: starts with a friendly greeting (non-empty, no bullet)
    if lines and not lines[0].startswith("-"):
        scores["greeting"] += 1

    # Rule 2: has bullet points
    if any(line.strip().startswith("-") for line in lines):
        scores["bullets"] += 1

    # Rule 3: ends with the closing line
    if "anything else" in response.lower():
        scores["closing"] += 1

print("Style compliance (unseen queries):\n")
for rule, count in scores.items():
    bar = "#" * (count * 20 // total)
    print(f"  {rule:<12} {count}/{total} ({100*count/total:.0f}%) {bar}")
print(f"\n  {'overall':<12} {sum(scores.values())}/{total*3} ({100*sum(scores.values())/(total*3):.0f}%)")

## 4.8 Save the Adapter

In [ ]:
adapter_path = "./adapter"
model.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

## 4.9 The Full Picture — Chapters 2, 3, 4 Compared

| | Ch2: Classify | Ch3: Extract | Ch4: Instruct |
|---|---|---|---|
| **Output type** | 1 word from set | JSON object | Multi-paragraph |
| **What model learns** | Categorization | Field extraction | Style + structure |
| **Prompt format** | `Classify: X\nLabel:` | `Extract: X\nJSON:` | `Query: X\nResponse:` |
| **`max_new_tokens`** | 3 | 60 | 150 |
| **Fine-tuning code** | **Identical** | **Identical** | **Identical** |

**The only thing that changed across all three chapters is the training data.** The model architecture, LoRA config, tokenization, collation, and training loop are exactly the same. This is the fundamental insight: fine-tuning is about showing the model examples of the behavior you want — the code infrastructure barely changes.